# DNABERT-2 with Real Genomic DNA

Tests DNABERT-2 on **real genomic sequences** (human chr22).

## Purpose

- **Synthetic DNA (other notebook)**: Tests pure architectural properties
- **Real DNA (this notebook)**: Ecological validity for BERT-style DNA models

DNABERT-2 uses BPE tokenization (vs. k-mer or character-level), giving a
unique perspective on how tokenization strategy affects geometric stability
on real genomic data.

## Setup Instructions

1. Upload `evaluation_harness.py` and `perturbation_protocol.py`
2. Change Runtime to GPU (Runtime > Change runtime type > GPU)
3. Run cells in order

---

In [ ]:
# Install Dependencies

print("Installing core dependencies...")
!pip install -q torch transformers shesha-geometry matplotlib seaborn pandas einops

print("\nInstalling triton (for DNABERT-2 FlashAttention)...")
!pip install -q triton

import transformers, torch
print(f"\ntransformers {transformers.__version__}")
print(f"torch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# --- Patch transformers 5.x for DNABERT-2 compatibility ---
# DNABERT-2's custom BertConfig (configuration_bert.py) expects certain
# attributes on PretrainedConfig that were removed in transformers 5.x.
from transformers import PretrainedConfig
import transformers.modeling_utils as _mu

_config_defaults = {
    'pad_token_id': None,
    'bos_token_id': None,
    'eos_token_id': None,
    'is_decoder': False,
    'add_cross_attention': False,
    'chunk_size_feed_forward': 0,
    'is_encoder_decoder': False,
    'tie_word_embeddings': True,
}
for attr, default in _config_defaults.items():
    if not hasattr(PretrainedConfig, attr):
        setattr(PretrainedConfig, attr, default)
        print(f"Patched config default: {attr}={default}")

# Patch all_tied_weights_keys for models that don't define it
if not getattr(_mu, '_dnabert2_patched', False):
    orig_mark = _mu.PreTrainedModel.mark_tied_weights_as_initialized
    def safe_mark(self):
        if not hasattr(self, 'all_tied_weights_keys'):
            self.all_tied_weights_keys = {}
        return orig_mark(self)
    _mu.PreTrainedModel.mark_tied_weights_as_initialized = safe_mark

    orig_finalize = _mu.PreTrainedModel._finalize_load_state_dict
    @staticmethod
    def safe_finalize(model, load_config, load_info):
        if not hasattr(model, 'all_tied_weights_keys'):
            model.all_tied_weights_keys = {}
        orig_tie = model.tie_weights
        def _patched_tie(missing_keys=None, recompute_mapping=False, **kwargs):
            return orig_tie()
        model.tie_weights = _patched_tie
        return orig_finalize(model, load_config, load_info)
    _mu.PreTrainedModel._finalize_load_state_dict = safe_finalize
    _mu._dnabert2_patched = True
    print("Patched: all_tied_weights_keys / tie_weights")

# Patch 3: Disable meta-device initialization
# transformers 5.x + accelerate uses init_empty_weights() to create models
# on a 'meta' device first (lazy tensors). DNABERT-2's custom code does real
# tensor ops during __init__ which crashes on meta tensors.
# Replace init_empty_weights with a no-op context manager.
from contextlib import contextmanager

@contextmanager
def _no_meta_init(**kwargs):
    """No-op replacement for init_empty_weights -- tensors init on CPU."""
    yield

try:
    import accelerate
    accelerate.init_empty_weights = _no_meta_init
    print("Patched: accelerate.init_empty_weights (disabled meta init)")
except ImportError:
    pass

# Also patch wherever transformers caches the reference
for mod_name in ['transformers.modeling_utils', 'transformers.integrations.accelerate']:
    import sys
    mod = sys.modules.get(mod_name)
    if mod and hasattr(mod, 'init_empty_weights'):
        mod.init_empty_weights = _no_meta_init
        print(f"Patched: {mod_name}.init_empty_weights")

print("\nDone!")

In [ ]:
# Configuration

import sys, os
sys.path.insert(0, '.')
PHASE = 'full'

OUTPUT_BASE = './results/dnabert2_realdna_experiment/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR = OUTPUT_BASE + 'cache'

CONFIG = {
    'quick': {
        'n_sequences': 500,
        'seq_length': 1000,
        'models': [
            'zhihan1996/DNABERT-2-117M',
        ],
        'batch_size': 8,
        'n_bootstrap': 0,
        'snp_rates': [0.01, 0.05, 0.10],
    },
    'full': {
        'n_sequences': 10000,
        'seq_length': 1000,
        'models': [
            'zhihan1996/DNABERT-2-117M',
        ],
        'batch_size': 8,
        'n_bootstrap': 5,
        'snp_rates': [0.01, 0.02, 0.05, 0.10],
    },
}

config = CONFIG[PHASE]
print(f"Phase: {PHASE.upper()}")
print(f"Architecture: DNABERT-2 (BERT + BPE)")
print(f"Data: Real human genomic DNA (chr22)")
print(f"Sequences: {config['n_sequences']}")
print(f"Models: {len(config['models'])}")

In [ ]:
# Load Real Genomic DNA

import urllib.request
import gzip
import os
import numpy as np

CHR22_URL = 'https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr22.fa.gz'
VALID_BASES = set('ACGT')


def download_and_sample_genomic_dna(n_sequences, seq_length, seed=320):
    cache_file = f'{CACHE_DIR}/chr22_sample_{n_sequences}_{seq_length}.txt'

    if os.path.exists(cache_file):
        print(f"Loading cached genomic sequences: {cache_file}")
        with open(cache_file) as f:
            sequences = [line.strip() for line in f if line.strip()]
        print(f"Loaded {len(sequences)} cached sequences")
        return sequences

    print("Downloading human chr22 (~50MB, first run only)...")
    os.makedirs(CACHE_DIR, exist_ok=True)

    with urllib.request.urlopen(CHR22_URL) as response:
        with gzip.GzipFile(fileobj=response) as f:
            lines = f.read().decode('utf-8').split('\n')
            chr22_seq = ''.join(line.strip() for line in lines[1:] if line.strip())

    print(f"Downloaded chr22 ({len(chr22_seq):,} bp)")

    buffer_factor = 1.5
    max_attempts = int(n_sequences * buffer_factor)
    rng = np.random.default_rng(seed)
    sequences = []

    for attempt in range(max_attempts):
        start = rng.integers(0, len(chr22_seq) - seq_length)
        seq = chr22_seq[start:start + seq_length].upper()

        n_count = sum(1 for c in seq if c not in VALID_BASES)
        if n_count < seq_length * 0.1:
            seq_clean = ''.join(
                c if c in VALID_BASES else rng.choice(['A', 'C', 'G', 'T'])
                for c in seq
            )
            sequences.append(seq_clean)

        if len(sequences) >= n_sequences:
            break

    sequences = sequences[:n_sequences]
    print(f"Sampled {len(sequences)} clean sequences")

    with open(cache_file, 'w') as f:
        f.write('\n'.join(sequences))
    print(f"Cached to {cache_file}")
    return sequences


sequences = download_and_sample_genomic_dna(
    config['n_sequences'], config['seq_length'], seed=320,
)

print(f"\nSequence stats:")
print(f"  Count: {len(sequences)}")
print(f"  Length: {len(sequences[0])}")
print(f"  Sample: {sequences[0][:60]}...")

total_bases = sum(len(s) for s in sequences)
print(f"\nBase composition:")
for base in 'ACGT':
    count = sum(s.count(base) for s in sequences)
    print(f"  {base}: {count/total_bases*100:.1f}%")

In [ ]:
# DNA Perturbation Suite

from dataclasses import dataclass, field

DNA_BASES = ['A', 'C', 'G', 'T']
COMPLEMENT = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}

@dataclass
class PerturbedSet:
    name: str
    category: str
    sequences: list
    params: dict = field(default_factory=dict)
    description: str = ''


def mutate_dna(sequence, mutation_rate, rng):
    seq = list(sequence)
    valid_positions = [i for i, c in enumerate(seq) if c in DNA_BASES]
    n_mutations = max(1, int(len(valid_positions) * mutation_rate))
    positions = rng.choice(valid_positions, size=min(n_mutations, len(valid_positions)), replace=False)
    for pos in positions:
        original = seq[pos]
        alternatives = [b for b in DNA_BASES if b != original]
        seq[pos] = rng.choice(alternatives)
    return ''.join(seq)


def reverse_complement(sequence):
    return ''.join(COMPLEMENT.get(b, b) for b in reversed(sequence))


class DNAPerturbationSuite:
    def __init__(self, seed=320, snp_rates=None, include_rc=True):
        self.rng = np.random.default_rng(seed)
        self.snp_rates = snp_rates or [0.01, 0.02, 0.05, 0.10]
        self.include_rc = include_rc

    def run_all(self, sequences):
        results = {}
        for rate in self.snp_rates:
            name = f"snp_{int(rate * 100)}pct"
            perturbed = [mutate_dna(s, rate, self.rng) for s in sequences]
            results[name] = PerturbedSet(
                name=name, category='snp', sequences=perturbed,
                params={'mutation_rate': rate},
                description=f'SNP mutations at {rate*100:.0f}% rate',
            )
        if self.include_rc:
            results['reverse_complement'] = PerturbedSet(
                name='reverse_complement', category='rc',
                sequences=[reverse_complement(s) for s in sequences],
                params={}, description='Reverse complement transformation',
            )
        return results


suite = DNAPerturbationSuite(seed=320, snp_rates=config['snp_rates'])

test_perturbed = suite.run_all(sequences[:3])
for name, pset in test_perturbed.items():
    dists = [
        sum(a != b for a, b in zip(o, p)) / len(o)
        for o, p in zip(sequences[:3], pset.sequences)
    ]
    print(f"  {name}: mean_hamming={np.mean(dists):.4f}")
print("Perturbation suite ready")

In [ ]:
# DNABERT-2 Model Wrapper

import torch
import gc
import importlib
from transformers import AutoTokenizer, AutoConfig
from huggingface_hub import hf_hub_download


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        mem = torch.cuda.memory_allocated() / 1e9
        print(f"  GPU memory after cleanup: {mem:.2f} GB")


def make_dnabert2_embedding_fn(model_name, batch_size=8):
    print(f"Loading {model_name}...")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"  Device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    # --- Manual loading: bypass AutoModel.from_pretrained entirely ---
    # transformers 5.x + accelerate forces meta-tensor init inside from_pretrained.
    # DNABERT-2's custom bert_layers.py does real tensor ops in __init__ and crashes.
    # Resolve the model CLASS via HF's dynamic module utils, instantiate
    # directly on CPU, then load the state dict manually.
    import sys, os, re
    from transformers.dynamic_module_utils import get_class_from_dynamic_module

    # Step 1: Load config
    config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)

    # Step 2: Patch flash_attn_triton.py for Triton >= 3.x compatibility
    # DNABERT-2's bundled flash_attn_triton.py uses the old API:
    #   tl.dot(q, k, trans_b=True)
    # Triton 3.x removed trans_b; the new API is:
    #   tl.dot(q, tl.trans(k))
    # We patch the cached source file BEFORE it gets imported.
    flash_path = hf_hub_download(model_name, 'flash_attn_triton.py')
    with open(flash_path, 'r') as f:
        source = f.read()
    if 'trans_b=True' in source:
        source = re.sub(
            r'tl\.dot\((\w+),\s*(\w+),\s*trans_b\s*=\s*True\)',
            r'tl.dot(\1, tl.trans(\2))',
            source
        )
        with open(flash_path, 'w') as f:
            f.write(source)
        # Clear any previously-cached imports so the patched file is used
        for key in list(sys.modules.keys()):
            if 'flash_attn_triton' in key or 'bert_layers' in key:
                del sys.modules[key]
        print("  Patched flash_attn_triton.py for Triton >= 3.x")
    else:
        print("  flash_attn_triton.py already compatible")

    # Step 3: Resolve the custom BertModel class via HF's module system
    # This downloads + properly registers bert_layers.py (and its deps) as modules
    auto_map = getattr(config, 'auto_map', {})
    class_ref = auto_map.get('AutoModel', auto_map.get('AutoModelForMaskedLM', 'bert_layers.BertModel'))
    print(f"  Resolving class: {class_ref}")
    BertModel = get_class_from_dynamic_module(class_ref, model_name)
    print(f"  Resolved: {BertModel.__module__}.{BertModel.__name__}")

    # Step 4: Create model on CPU -- no meta tensors!
    print("  Initializing model on CPU...")
    model = BertModel(config)

    # Step 5: Load pretrained weights
    weight_file = hf_hub_download(model_name, 'pytorch_model.bin')
    print("  Loading pretrained weights...")
    state_dict = torch.load(weight_file, map_location='cpu', weights_only=False)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        print(f"  Note: {len(missing)} missing keys (normal for base model)")
    if unexpected:
        print(f"  Note: {len(unexpected)} unexpected keys")

    model = model.to(device).eval()

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    if torch.cuda.is_available():
        mem = torch.cuda.memory_allocated() / 1e9
        print(f"  Params: {n_params:.1f}M | GPU mem: {mem:.2f} GB")
    else:
        print(f"  Params: {n_params:.1f}M")

    max_length = tokenizer.model_max_length
    if max_length > 100000:
        max_length = 512
    print(f"  Max token length: {max_length}")

    @torch.no_grad()
    def embed(sequences):
        embeddings = []
        n_batches = (len(sequences) + batch_size - 1) // batch_size

        for i in range(0, len(sequences), batch_size):
            batch = sequences[i:i + batch_size]
            batch_idx = i // batch_size + 1

            if batch_idx % 20 == 0 or batch_idx == n_batches:
                print(f"    Batch {batch_idx}/{n_batches}", end='\r')

            tokens = tokenizer(
                batch, return_tensors='pt', padding=True,
                truncation=True, max_length=max_length,
            )
            tokens = {k: v.to(device) for k, v in tokens.items()}

            outputs = model(**tokens)
            hidden = outputs[0]

            attention_mask = tokens['attention_mask'].unsqueeze(-1)
            pooled = (hidden * attention_mask).sum(dim=1) / attention_mask.sum(dim=1).clamp(min=1)
            embeddings.append(pooled.cpu().numpy())

        print()
        return np.concatenate(embeddings, axis=0)

    return embed, model, tokenizer, n_params

print("DNABERT-2 wrapper ready")

In [ ]:
# Evaluation Harness

from evaluation_harness import StabilityHarness

harness = StabilityHarness(
    window_size=0,
    metric='cosine',
    n_splits=30,
    seed=320,
    max_samples=2500,
    n_bootstrap=config['n_bootstrap'],
)

print(f"Harness configured (bootstrap={config['n_bootstrap']})")

In [ ]:
# Run Experiment

import os
import time
import pandas as pd
from evaluation_harness import ModelReport

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print('=' * 70)
print(f'DNABERT-2 + REAL GENOMIC DNA -- Phase: {PHASE.upper()}')
print('=' * 70)

reports = []
model_param_counts = []
all_detailed_rows = []

for model_idx, model_name in enumerate(config['models']):
    print(f"\n{'='*70}")
    print(f"[{model_idx+1}/{len(config['models'])}] {model_name}")
    print('=' * 70)

    start_time = time.time()

    embed_fn, model_obj, tokenizer_obj, n_params_m = make_dnabert2_embedding_fn(
        model_name, config['batch_size']
    )
    model_param_counts.append(n_params_m)

    print('\nGenerating perturbations...')
    perturbed_sets = suite.run_all(sequences)

    safe_name = model_name.replace('/', '_').replace('-', '_') + '_realdna'
    cache_clean = f'{CACHE_DIR}/{safe_name}_clean.npy'

    if os.path.exists(cache_clean):
        print(f'Loading cached: {cache_clean}')
        embeddings_clean = np.load(cache_clean)
    else:
        print(f'Computing clean embeddings ({len(sequences)} sequences)...')
        embeddings_clean = embed_fn(sequences)
        np.save(cache_clean, embeddings_clean)
        print(f'  Cached to {cache_clean}')
    print(f'  Shape: {embeddings_clean.shape}')

    perturbed_embeddings = {}
    for pert_name, pset in perturbed_sets.items():
        cache_pert = f'{CACHE_DIR}/{safe_name}_{pert_name}.npy'
        if os.path.exists(cache_pert):
            print(f'  Loading cached: {pert_name}')
            perturbed_embeddings[pert_name] = np.load(cache_pert)
        else:
            print(f'  Embedding: {pert_name}...')
            perturbed_embeddings[pert_name] = embed_fn(pset.sequences)
            np.save(cache_pert, perturbed_embeddings[pert_name])

    del model_obj, tokenizer_obj
    cleanup_gpu()

    print('\nRunning Shesha evaluation...')
    report = harness.evaluate_all_perturbations(
        model_name=model_name,
        embeddings_clean=embeddings_clean,
        perturbed_dict=perturbed_embeddings,
    )
    reports.append(report)

    short_name = model_name.split('/')[-1]
    for pert_name, metrics in report.perturbation_breakdown().items():
        all_detailed_rows.append({
            'model': short_name,
            'size_M': round(n_params_m, 1),
            'perturbation': pert_name,
            'rdm_similarity': metrics.get('rdm_similarity_score', 0),
            'rdm_drift': metrics.get('rdm_drift', 0),
            'pert_stability': metrics.get('perturbation_stability_score', 0),
            'pert_magnitude': metrics.get('perturbation_magnitude', 0),
            'composite': metrics.get('composite_stability', 0),
        })

    elapsed = time.time() - start_time
    summary = report.summary()
    print(f'\nCompleted in {elapsed/60:.1f} min')
    print(f'  Composite Stability: {summary["mean_composite_stability"]:.4f}')
    print(f'  RDM Similarity:      {summary["mean_rdm_similarity_score"]:.4f}')

df_detailed = pd.DataFrame(all_detailed_rows)
detailed_path = f'{RESULTS_DIR}/dnabert2_realdna_{PHASE}_detailed.csv'
df_detailed.to_csv(detailed_path, index=False)
print(f'\nPer-perturbation results:')
print(df_detailed.to_string(index=False))
print(f'\nSaved to {detailed_path}')

print('\n' + '=' * 70)
print('EXPERIMENT COMPLETE')
print('=' * 70)

In [ ]:
# Results & Visualization

from evaluation_harness import compare_models
import matplotlib.pyplot as plt

comparison = compare_models(reports, output_dir=RESULTS_DIR)

print('\n' + '=' * 70)
print('DNABERT-2 + REAL DNA SUMMARY')
print('=' * 70 + '\n')

for model_name, data in comparison['models'].items():
    s = data['summary']
    short = model_name.split('/')[-1]
    print(f'{short}:')
    print(f'  Composite Stability:  {s["mean_composite_stability"]:.4f} +/- {s["std_composite_stability"]:.4f}')
    print()

# Per-perturbation breakdown
fig, ax = plt.subplots(figsize=(10, 6))
perts = df_detailed['perturbation'].tolist()
composites = df_detailed['composite'].tolist()
colors = ['tab:blue' if 'snp' in p else 'tab:orange' for p in perts]
ax.barh(perts, composites, color=colors)
ax.set_xlabel('Composite Stability')
ax.set_title('DNABERT-2 on Real DNA: Stability by Perturbation', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/dnabert2_realdna_scaling_{PHASE}.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cross-Experiment Comparison

import matplotlib.pyplot as plt
import pandas as pd
import glob

fig, ax = plt.subplots(figsize=(10, 6))

# DNABERT-2 (single point)
dnabert2_composite = df_detailed['composite'].mean()
ax.scatter(model_param_counts, [dnabert2_composite], s=200, c='tab:green',
           marker='P', zorder=5, label='DNABERT-2 (BERT/BPE)')

# Load other architectures' results
for label, pattern, color, marker in [
    ('NT (Transformer)', '**/nt_*_detailed.csv', 'tab:orange', 's-'),
    ('ESM-2 (Transformer)', '**/esm2_*_detailed.csv', 'tab:red', 'o-'),
    ('Caduceus (SSM)', '**/caduceus_*_detailed.csv', 'tab:blue', 'D-'),
]:
    files = glob.glob(f'{RESULTS_DIR}/../../../{pattern}', recursive=True)
    if files:
        df = pd.read_csv(files[0])
        agg = df.groupby('size_M')['composite'].mean().reset_index()
        ax.plot(agg['size_M'], agg['composite'], marker,
                color=color, linewidth=2, markersize=10, label=label)

ax.set_xscale('log')
ax.set_xlabel('Parameters (M)', fontsize=12)
ax.set_ylabel('Composite Stability', fontsize=12)
ax.set_title('All Architectures: The Geometric Tax', fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/dnabert2_realdna_crossexp_{PHASE}.png', dpi=300, bbox_inches='tight')
plt.show()